# Fine-tuning Qwen3-1.7B avec Unsloth en local

Notebook local pour fine-tuner `unsloth/Qwen3-1.7B-unsloth-bnb-4bit` sur les datasets HF generes par le notebook EDA, ou lancer seulement le DPO depuis un modele deja fine-tune :

- SFT optionnel : `Maphe/medical-sft-5k` avec colonnes `instruction` et `response`
- DPO optionnel : `Maphe/medical-dpo-5k` avec colonnes `prompt`, `chosen`, `rejected`
- suivi TensorBoard des loss train/eval globales et des loss par tache QCM / texte libre
- GPU cible detecte : RTX 2080 Ti 8 Go, donc parametres conservateurs par defaut
- export local LoRA, puis push HF optionnel

Dans VS Code, selectionnez le kernel Python de la `.venv` du projet avant d'executer les cellules.

TensorBoard est lance par une cellule du notebook sur le port `8000`.

Le notebook n'installe pas de dependances dans une cellule : l'environnement local est pilote par `pyproject.toml` et `uv.lock`.

Le flux general est le suivant: verifier l'environnement local, se connecter a Hugging Face si besoin, charger le modele de base avec LoRA, preparer le dataset SFT, lancer l'entrainement, tester qualitativement les generations, puis sauvegarder ou publier les poids produits.

Le notebook est volontairement prudent pour tenir sur une machine locale avec peu de VRAM. Plusieurs variables de configuration permettent de basculer entre un smoke test rapide et un vrai entrainement complet sans reecrire les cellules.

## 1. Verification de l'environnement local

Dans VS Code, utilisez le kernel Python de la `.venv` du projet. La cellule suivante verifie que le kernel pointe vers `.venv`, que les versions Unsloth sont alignees, que TensorBoard est disponible, et que CUDA voit le GPU.

Comme dans l'autre notebook, la cellule remonte automatiquement jusqu'a la racine du depot pour retrouver `pyproject.toml`. Cela evite les erreurs de chemins relatifs quand le notebook est lance depuis un sous-dossier ou une session distante.

La liste des packages sert de diagnostic rapide avant de lancer un fine-tuning couteux. Si une dependance critique apparait comme `MISSING` ou avec une version inattendue, il vaut mieux corriger l'environnement tout de suite plutot que de depanner au milieu d'un entrainement.

In [1]:
import importlib.metadata as md
import os
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

print(f"Python executable : {sys.executable}")
print(f"Repo root         : {repo_root}")
print(f"HF endpoint       : {os.environ.get('HF_ENDPOINT', 'https://huggingface.co')}")

expected_packages = [
    "torch", "transformers", "trl", "unsloth", "unsloth_zoo",
    "bitsandbytes", "accelerate", "peft", "datasets", "huggingface_hub",
    "setuptools", "tensorboard",
]
for package in expected_packages:
    try:
        print(f"{package:16s}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package:16s}: MISSING")

if ".venv" not in sys.executable:
    print("\nAttention: le kernel ne semble pas utiliser la .venv uv du projet.")
    print("Dans VS Code: Python: Select Interpreter, puis choisissez la .venv du depot.")


Python executable : /storage/user/zmxw1768/fine_tuning_oc/.venv/bin/python
Repo root         : /storage/user/zmxw1768/fine_tuning_oc
HF endpoint       : https://huggingface.co
torch           : 2.10.0
transformers    : 4.57.6
trl             : 0.24.0
unsloth         : 2026.5.2
unsloth_zoo     : 2026.5.1
bitsandbytes    : 0.49.2
accelerate      : 1.13.0
peft            : 0.19.1
datasets        : 4.3.0
huggingface_hub : 0.36.2
setuptools      : 80.10.2
tensorboard     : 2.20.0


## 2. Connexion Hugging Face et configuration

Cette partie separe volontairement l'authentification Hugging Face de la configuration technique du run. Le login n'est necessaire que si vous voulez pousser un modele ou acceder a des ressources privees, mais il est pratique de le faire en debut de notebook pour eviter une interruption plus tard.

La grande cellule de configuration centralise ensuite les variables d'environnement, les hyperparametres LoRA, les options SFT et DPO, les chemins de sortie, les graines aleatoires et les parametres TensorBoard. C'est le point d'entree principal a modifier pour adapter l'entrainement a votre GPU, a la taille du dataset ou au type de run souhaite.

In [2]:

from huggingface_hub import login

login()  # Coller un token HF avec droit write si vous voulez push le modele.


In [3]:
from __future__ import annotations

import inspect
import math
import os
import random
import sys
from typing import Any
from time import perf_counter

# A definir avant l'import Unsloth.
os.environ.setdefault("HF_ENDPOINT", "https://huggingface.co")
os.environ.setdefault("UNSLOTH_STABLE_DOWNLOADS", "1")

# Si un import Unsloth a echoue plus tot dans ce kernel, Python peut garder des
# modules partiellement charges. On les purge avant le vrai import.
for module_name in list(sys.modules):
    if module_name == "unsloth" or module_name.startswith("unsloth."):
        del sys.modules[module_name]

# Unsloth doit etre importe avant transformers / trl pour appliquer ses patchs.
from unsloth import FastModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

import numpy as np
import torch
from datasets import Dataset, DatasetDict, load_dataset
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from transformers import DataCollatorForLanguageModeling, TrainerCallback
from trl import DPOConfig, DPOTrainer, SFTConfig, SFTTrainer

# ---------------------------------------------------------------------
# Configuration projet local
# ---------------------------------------------------------------------
HF_USERNAME = "Maphe"
SFT_DATASET_ID = f"{HF_USERNAME}/medical-sft-5k"
DPO_DATASET_ID = f"{HF_USERNAME}/medical-dpo-5k"
OUTPUT_HUB_ID = f"{HF_USERNAME}/qwen3-1.7b-medical-finetuned"

BASE_MODEL_NAME = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit"
RUN_SFT = False
RUN_DPO = True

# Pour lancer seulement le DPO depuis un modele deja fine-tune:
# - mettre RUN_SFT = False et RUN_DPO = True
# - renseigner FINETUNED_MODEL_NAME avec un chemin local ou repo HF
# - garder FINETUNED_MODEL_IS_LORA = True si c'est un adapter LoRA Unsloth/PEFT
FINETUNED_MODEL_NAME: str | None = "./qwen3-medical-lora"  # ex: "./qwen3-medical-lora" ou "Maphe/qwen3-1.7b-medical-lora"
FINETUNED_MODEL_IS_LORA = True
MODEL_NAME = FINETUNED_MODEL_NAME if (FINETUNED_MODEL_NAME and not RUN_SFT) else BASE_MODEL_NAME
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = True

LORA_RANK = 16
LORA_ALPHA = 16

SFT_EPOCHS = 2
SFT_BATCH_SIZE = 32
SFT_GRAD_ACCUM = 16
SFT_EVAL_BATCH_SIZE = 1
SFT_LR = 2e-4
SFT_MAX_STEPS: int | None = None  # Mettre None pour lancer le vrai fine-tuning.
SFT_SMOKE_ROWS: int | None = None  # Mettre None pour utiliser tout le dataset SFT.
SFT_EVAL_ROWS: int | None = None  # Mettre une valeur basse pour accelerer l'eval.
SFT_EVAL_STEPS = 1
ENABLE_TASK_LOSS_TENSORBOARD = True
SFT_TASK_LOSS_ROWS: int | None = 128  # Echantillon par tache pour les courbes TensorBoard si activees.
TENSORBOARD_LOG_DIR = "./sft_output/tensorboard"
TENSORBOARD_PORT = 8000

DPO_EPOCHS = 1
DPO_BATCH_SIZE = 4 # DPO est plus gourmand en VRAM que SFT, on utilise une batch plus petite et plus de grad_accum.
DPO_GRAD_ACCUM = 8 # 
DPO_LR = 5e-5
DPO_BETA = 0.1
DPO_MAX_STEPS: int | None = None

LOCAL_LORA_OUTPUT_DIR = "./qwen3-medical-dpo-lora" if RUN_DPO and not RUN_SFT else "./qwen3-medical-lora"
PUSH_TO_HUB = False
PRIVATE_HUB_REPO = True
SEED = 42

SYSTEM_PROMPT = (
    "Tu es un assistant medical expert. "
    "Reponds de maniere claire, factuelle et structuree. "
    "Si la question est en anglais, reponds en anglais."
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU CUDA detecte. Dans Colab: Runtime > Change runtime type > GPU.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")
print(f"bf16 supporte: {is_bfloat16_supported()} (RTX 2080 Ti: False attendu, fp16 sera utilise)")
print(f"Modele charge      : {MODEL_NAME}")
print(f"RUN_SFT / RUN_DPO  : {RUN_SFT} / {RUN_DPO}")
if not RUN_SFT and RUN_DPO and FINETUNED_MODEL_NAME is None:
    print("Attention: DPO-only actif sans FINETUNED_MODEL_NAME; le DPO partira du modele de base.")
print(f"Sortie LoRA locale : {LOCAL_LORA_OUTPUT_DIR}")
print(f"TensorBoard log_dir: {TENSORBOARD_LOG_DIR}")
print(f"TensorBoard port   : {TENSORBOARD_PORT}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: NVIDIA GeForce RTX 3090
CUDA: 12.8
bf16 supporte: True (RTX 2080 Ti: False attendu, fp16 sera utilise)
Modele charge      : ./qwen3-medical-lora
RUN_SFT / RUN_DPO  : False / True
Sortie LoRA locale : ./qwen3-medical-dpo-lora
TensorBoard log_dir: ./sft_output/tensorboard
TensorBoard port   : 8000


## 3. TensorBoard

Lancez cette cellule avant l'entrainement pour suivre les courbes. TensorBoard est expose sur le port `8000` et lit les logs dans `./sft_output/tensorboard`.


In [4]:
%load_ext tensorboard
%tensorboard --logdir ./sft_output/tensorboard --host 0.0.0.0 --port 8000


## 3. Helpers de compatibilite TRL

Ces helpers evitent de casser le notebook quand TRL change quelques noms d'arguments (`tokenizer` / `processing_class`, `max_seq_length` / `max_length`).

L'idee est de rendre le notebook plus robuste aux petites differences d'API entre versions de `trl` et `transformers`. Au lieu de figer une signature unique, ces fonctions inspectent les classes disponibles localement et ne passent que les arguments supportes.

Cette couche de compatibilite limite les erreurs du type `unexpected keyword argument` qui sont frequentes quand on reprend un notebook quelques semaines plus tard dans un environnement legerement different.


In [5]:
def filtered_kwargs(cls_or_fn: Any, values: dict[str, Any]) -> dict[str, Any]:
    """Garde uniquement les kwargs supportes par la version locale."""
    # Evite les erreurs si une version locale retire ou renomme certains parametres.
    signature = inspect.signature(cls_or_fn)
    return {key: value for key, value in values.items() if key in signature.parameters}


def make_sft_config(**overrides: Any) -> SFTConfig:
    # Construit une config SFT raisonnable pour une machine locale avec VRAM limitee.
    values = {
        "output_dir": "./sft_output",
        "overwrite_output_dir": True,
        "num_train_epochs": SFT_EPOCHS,
        "max_steps": SFT_MAX_STEPS if SFT_MAX_STEPS is not None else -1,
        "per_device_train_batch_size": SFT_BATCH_SIZE,
        "per_device_eval_batch_size": SFT_EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": SFT_GRAD_ACCUM,
        "learning_rate": SFT_LR,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.03,
        "fp16": not is_bfloat16_supported(),
        "bf16": is_bfloat16_supported(),
        "logging_steps": 2,
        "logging_first_step": True,
        "eval_steps": SFT_EVAL_STEPS,
        "eval_strategy": "steps",
        "evaluation_strategy": "steps",
        "do_eval": True,
        "prediction_loss_only": True,
        "save_strategy": "epoch",
        "seed": SEED,
        "optim": "adamw_8bit",
        "dataset_text_field": "text",
        "packing": False,
        "padding_free": False,
        "remove_unused_columns": False,
        "report_to": ["tensorboard"],
        "logging_dir": TENSORBOARD_LOG_DIR,
        "dataset_num_proc": min(4, os.cpu_count() or 1),
        "dataloader_num_workers": 0,
    }

    # TRL recent utilise max_length; des versions plus anciennes utilisent max_seq_length.
    config_signature = inspect.signature(SFTConfig)
    if "max_length" in config_signature.parameters:
        values["max_length"] = MAX_SEQ_LENGTH
    else:
        values["max_seq_length"] = MAX_SEQ_LENGTH

    values.update(overrides)
    return SFTConfig(**filtered_kwargs(SFTConfig, values))


def make_dpo_config(**overrides: Any) -> DPOConfig:
    # Configuration separee pour le DPO, plus simple et desactivee par defaut.
    values = {
        "output_dir": "./dpo_output",
        "overwrite_output_dir": True,
        "num_train_epochs": DPO_EPOCHS,
        "max_steps": DPO_MAX_STEPS if DPO_MAX_STEPS is not None else -1,
        "per_device_train_batch_size": DPO_BATCH_SIZE,
        "gradient_accumulation_steps": DPO_GRAD_ACCUM,
        "learning_rate": DPO_LR,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.1,
        "fp16": not is_bfloat16_supported(),
        "bf16": is_bfloat16_supported(),
        "logging_steps": 2,
        "save_strategy": "epoch",
        "seed": SEED,
        "optim": "adamw_8bit",
        "beta": DPO_BETA,
        "max_length": MAX_SEQ_LENGTH,
        "max_prompt_length": MAX_SEQ_LENGTH // 2,
        "padding_free": False,
        "remove_unused_columns": False,
        "report_to": "none",
        "dataset_num_proc": 1,
        "dataloader_num_workers": 0,
    }
    values.update(overrides)
    return DPOConfig(**filtered_kwargs(DPOConfig, values))


def trainer_tokenizer_kwarg(trainer_cls: Any, tokenizer: Any) -> dict[str, Any]:
    # TRL a renomme ce parametre selon les versions; on choisit celui qui existe.
    params = inspect.signature(trainer_cls.__init__).parameters
    if "processing_class" in params:
        return {"processing_class": tokenizer}
    if "tokenizer" in params:
        return {"tokenizer": tokenizer}
    return {}


## 4. Chargement du modele Qwen3 avec LoRA Unsloth

Le checkpoint par defaut est la variante Unsloth 4-bit pour limiter la VRAM locale. Sur RTX 2080 Ti, le notebook utilise `fp16`, `MAX_SEQ_LENGTH=1024`, `batch_size=1` et LoRA rank 16 par defaut.

La cellule suivante charge d'abord le modele de base quantifie, puis applique les adapters LoRA uniquement sur les modules cibles de l'attention et du MLP. C'est ce qui permet d'entrainer un grand modele sur une machine plus modeste sans mettre a jour tous les poids.

Le `device_map="sequential"` et les reglages de padding sont conservateurs: ils privilegient la stabilite et la compatibilite locale plutot qu'une recherche agressive de performances.

In [6]:

loading_existing_lora = bool(FINETUNED_MODEL_NAME and not RUN_SFT and FINETUNED_MODEL_IS_LORA)

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=False,
    full_finetuning=False,
    dtype=None,
    device_map="sequential",
)

# Certains tokenizers n'ont pas de pad_token par defaut; on l'aligne sur eos pour le batching.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Le padding a droite simplifie les collators classiques utilises pour le SFT/DPO.
if getattr(tokenizer, "padding_side", None) != "right":
    tokenizer.padding_side = "right"

if loading_existing_lora:
    FastModel.for_training(model)
    print(f"Adapter LoRA deja fine-tune charge depuis: {MODEL_NAME}")
else:
    # On ajoute un adapter LoRA entrainable sur le modele de base ou un modele fusionne.
    model = FastModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        max_seq_length=MAX_SEQ_LENGTH,
    )

model.print_trainable_parameters()


==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 24.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Adapter LoRA deja fine-tune charge depuis: ./qwen3-medical-lora
trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


## 5. Chargement et preparation du dataset SFT optionnel

Cette section transforme le dataset source en exemples directement consommables par `SFTTrainer` quand `RUN_SFT = True`. Elle valide d'abord le schema, nettoie les textes, reconstruit les conversations au format chat Qwen3, puis fabrique au besoin un split de validation deterministe.

La meme cellule prepare aussi les jeux servant au suivi TensorBoard par type de tache. Cela permet de distinguer une evolution differente entre QCM et texte libre, ce qui est souvent plus instructif qu'une loss globale unique.

In [7]:
def require_columns(dataset: Dataset, columns: set[str], dataset_id: str) -> None:
    missing = columns - set(dataset.column_names)
    if missing:
        raise ValueError(
            f"Dataset {dataset_id} invalide. Colonnes manquantes: {sorted(missing)}. "
            f"Colonnes presentes: {dataset.column_names}"
        )


def clean_text(value: Any) -> str:
    # Normalise les espaces pour stabiliser les prompts et les exports.
    text = "" if value is None else str(value)
    return " ".join(text.strip().split())


def direct_answer(response: str) -> str:
    # Qwen3 non-thinking mode : bloc think vide, recommande par Qwen/Unsloth pour reponses directes.
    return f"<think>\n\n</think>\n\n{clean_text(response)}"


def task_group_from_example(example: dict[str, Any]) -> str:
    # Tente d'utiliser le champ task_type quand il existe, sinon retombe sur des heuristiques textuelles.
    task_type = clean_text(example.get("task_type", "")).lower()
    if task_type in {"mcq_single", "mcq", "qcm"}:
        return "qcm"
    if task_type in {"open_qa", "free_text", "texte_libre", "text"}:
        return "texte_libre"

    instruction = clean_text(example.get("instruction", "")).lower()
    if "options :" in instruction or "options:" in instruction or "option correcte" in instruction:
        return "qcm"
    return "texte_libre"


def format_sft_row(example: dict[str, Any]) -> dict[str, str]:
    # Recompose un exemple complet system/user/assistant au format attendu par Qwen3.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": clean_text(example["instruction"])},
        {"role": "assistant", "content": direct_answer(example["response"])},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    task_group = task_group_from_example(example)
    return {"text": text, "task_group": task_group}


def select_sft_splits(raw_dataset: DatasetDict | Dataset) -> tuple[Dataset, Dataset | None]:
    # Prefere un vrai split eval si disponible, sinon cree une validation deterministe.
    if isinstance(raw_dataset, DatasetDict):
        if "train" not in raw_dataset:
            raise ValueError(f"Split train introuvable. Splits disponibles: {list(raw_dataset)}")
        train_dataset = raw_dataset["train"]
        for split_name in ("validation", "eval", "test"):
            if split_name in raw_dataset:
                print(f"Split eval SFT utilise: {split_name}")
                return train_dataset, raw_dataset[split_name]
        print("Aucun split eval detecte: creation d'un split validation deterministe depuis train (10%).")
        split = train_dataset.train_test_split(test_size=0.1, seed=SEED, shuffle=True)
        return split["train"], split["test"]

    print("Dataset sans splits nommes: creation d'un split validation deterministe (10%).")
    split = raw_dataset.train_test_split(test_size=0.1, seed=SEED, shuffle=True)
    return split["train"], split["test"]


def strip_to_text_dataset(dataset: Dataset) -> Dataset:
    # Le trainer SFT n'a besoin que de la colonne text une fois le formatage termine.
    removable_columns = [column for column in dataset.column_names if column != "text"]
    return dataset.remove_columns(removable_columns) if removable_columns else dataset


def limit_dataset_rows(dataset: Dataset, max_rows: int | None) -> Dataset:
    # Utile pour accelerer les smoke tests ou limiter les jeux de suivi TensorBoard.
    if max_rows is None or len(dataset) <= max_rows:
        return dataset
    return dataset.select(range(max_rows))


def task_count_summary(dataset: Dataset) -> dict[str, int]:
    # Resume simple pour verifier l'equilibre QCM / texte libre dans chaque split.
    if "task_group" not in dataset.column_names:
        return {}
    counts: dict[str, int] = {}
    for task_group in dataset["task_group"]:
        counts[task_group] = counts.get(task_group, 0) + 1
    return counts


def response_loss_features(example: dict[str, Any]) -> dict[str, list[int]]:
    # Construit des labels ou seule la reponse assistant contribue a la loss.
    text = example["text"]
    tokenized = tokenizer(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )
    input_ids = tokenized["input_ids"]
    labels = list(input_ids)

    response_marker = "<|im_start|>assistant\n"
    response_start = text.find(response_marker)
    if response_start == -1:
        # Si le marqueur n'est pas retrouve, on masque tout pour eviter une supervision incorrecte.
        prompt_tokens = len(input_ids)
    else:
        prompt_text = text[: response_start + len(response_marker)]
        prompt_tokens = len(tokenizer(prompt_text, add_special_tokens=False)["input_ids"])

    prompt_tokens = min(prompt_tokens, len(labels))
    labels[:prompt_tokens] = [-100] * prompt_tokens
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


def tokenize_response_loss_dataset(dataset: Dataset) -> Dataset:
    # Filtre les exemples invalides qui ne laisseraient aucun token a entrainer.
    tokenized = dataset.map(response_loss_features, remove_columns=dataset.column_names)
    return tokenized.filter(lambda row: any(label != -100 for label in row["labels"]))


def make_task_loss_datasets(formatted_dataset: Dataset, split_name: str, max_rows: int | None) -> dict[str, Dataset]:
    # Prepare de petits sous-jeux specialises pour tracer une loss par type de tache.
    task_datasets: dict[str, Dataset] = {}
    for task_group in ("qcm", "texte_libre"):
        subset = formatted_dataset.filter(lambda row, group=task_group: row["task_group"] == group)
        if len(subset) == 0:
            continue
        subset = limit_dataset_rows(subset.shuffle(seed=SEED), max_rows)
        task_datasets[f"{split_name}_task/{task_group}_loss"] = tokenize_response_loss_dataset(subset)
    return task_datasets


def collate_response_loss_batch(features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
    # Collator minimaliste pour aligner input_ids, masques et labels sur une meme longueur.
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    max_length = max(len(feature["input_ids"]) for feature in features)
    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for feature in features:
        pad_length = max_length - len(feature["input_ids"])
        batch["input_ids"].append(feature["input_ids"] + [pad_token_id] * pad_length)
        batch["attention_mask"].append(feature["attention_mask"] + [0] * pad_length)
        batch["labels"].append(feature["labels"] + [-100] * pad_length)
    return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}


def estimate_response_loss(model: Any, dataset: Dataset) -> float:
    # Evalue la loss moyenne sur un sous-ensemble sans perturber l'etat d'entrainement du modele.
    if len(dataset) == 0:
        return float("nan")

    dataloader = DataLoader(
        dataset,
        batch_size=SFT_EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_response_loss_batch,
    )
    was_training = model.training
    model.eval()
    losses: list[torch.Tensor] = []
    amp_dtype = torch.bfloat16 if is_bfloat16_supported() else torch.float16

    try:
        with torch.inference_mode():
            for batch in dataloader:
                batch = {key: value.to(model.device) for key, value in batch.items()}
                with torch.autocast("cuda", dtype=amp_dtype, enabled=torch.cuda.is_available()):
                    outputs = model(**batch)
                loss = outputs.loss.detach().float().cpu()
                if torch.isfinite(loss):
                    losses.append(loss)
    finally:
        if was_training:
            model.train()

    if not losses:
        return float("nan")
    return float(torch.stack(losses).mean().item())


class TaskLossTensorBoardCallback(TrainerCallback):
    def __init__(self, log_dir: str, task_datasets: dict[str, Dataset]) -> None:
        # Ecrit des courbes separees pour suivre si une famille de taches diverge plus vite qu'une autre.
        self.writer = SummaryWriter(log_dir=str(Path(log_dir) / "task_loss"))
        self.task_datasets = task_datasets
        self.last_logged_step: int | None = None

    def on_evaluate(self, args: Any, state: Any, control: Any, model: Any | None = None, **kwargs: Any) -> Any:
        # Ne logue qu'une fois par step global et seulement sur le processus principal.
        if not state.is_world_process_zero or model is None or self.last_logged_step == state.global_step:
            return control

        self.last_logged_step = state.global_step
        logged: dict[str, float] = {}
        for tag, dataset in self.task_datasets.items():
            loss = estimate_response_loss(model, dataset)
            self.writer.add_scalar(tag, loss, state.global_step)
            logged[tag] = loss
        self.writer.flush()

        if logged:
            summary = ", ".join(f"{key}={value:.4f}" for key, value in logged.items())
            print(f"TensorBoard task loss step {state.global_step}: {summary}", flush=True)
        return control

    def on_train_end(self, args: Any, state: Any, control: Any, **kwargs: Any) -> Any:
        self.writer.flush()
        self.writer.close()
        return control


if RUN_SFT:
    sft_raw = load_dataset(SFT_DATASET_ID)
    sft_train_raw, sft_eval_raw = select_sft_splits(sft_raw)
    require_columns(sft_train_raw, {"instruction", "response"}, SFT_DATASET_ID)
    if sft_eval_raw is not None:
        require_columns(sft_eval_raw, {"instruction", "response"}, SFT_DATASET_ID)

    sft_train_formatted = sft_train_raw.map(format_sft_row, remove_columns=sft_train_raw.column_names)
    sft_train_formatted = sft_train_formatted.filter(lambda row: bool(row["text"].strip()))

    if sft_eval_raw is not None:
        sft_eval_formatted = sft_eval_raw.map(format_sft_row, remove_columns=sft_eval_raw.column_names)
        sft_eval_formatted = sft_eval_formatted.filter(lambda row: bool(row["text"].strip()))
    else:
        sft_eval_formatted = None

    # Affiche un apercu du prompt final pour verifier le format de conversation avant l'entrainement.
    print(f"SFT train charge: {len(sft_train_formatted)} lignes | taches: {task_count_summary(sft_train_formatted)}")
    if sft_eval_formatted is not None:
        print(f"SFT eval charge : {len(sft_eval_formatted)} lignes | taches: {task_count_summary(sft_eval_formatted)}")
    print(sft_train_formatted[0]["text"][:800])
else:
    sft_train_formatted = None
    sft_eval_formatted = None
    print("Preparation SFT ignoree: RUN_SFT = False. Le notebook peut enchainer directement sur le DPO.")


Preparation SFT ignoree: RUN_SFT = False. Le notebook peut enchainer directement sur le DPO.


## 6. Entrainement SFT optionnel

Le trainer journalise `train/loss` et `eval/loss` dans TensorBoard. Les courbes detaillees par tache sont desactivees par defaut pour eviter les longues pauses d'evaluation sur GPU 8 Go; activez `ENABLE_TASK_LOSS_TENSORBOARD = True` si vous en avez besoin.

Pour un smoke test rapide, renseignez `SFT_MAX_STEPS` et/ou `SFT_SMOKE_ROWS` dans la configuration. Pour le vrai fine-tuning, laissez `SFT_MAX_STEPS = None` et `SFT_SMOKE_ROWS = None`.

La cellule suivante prepare la variante de dataset effectivement utilisee pour le run, instancie `SFTConfig`, construit le `SFTTrainer`, applique le masquage des prompts pour n'entrainer que la reponse assistant, puis lance enfin l'entrainement. Les nombreux `print` intermediaires servent de points de controle avant d'engager du temps GPU.

In [8]:
if RUN_SFT:
    t0 = perf_counter()

    sft_train_formatted_for_run = sft_train_formatted
    if SFT_SMOKE_ROWS is not None:
        smoke_rows = min(SFT_SMOKE_ROWS, len(sft_train_formatted))
        sft_train_formatted_for_run = sft_train_formatted.select(range(smoke_rows))
        print(f"Mode smoke dataset: {smoke_rows}/{len(sft_train_formatted)} exemples train utilises.", flush=True)

    if sft_eval_formatted is not None:
        sft_eval_formatted_for_run = limit_dataset_rows(sft_eval_formatted.shuffle(seed=SEED), SFT_EVAL_ROWS)
    else:
        sft_eval_formatted_for_run = None

    sft_train_dataset = strip_to_text_dataset(sft_train_formatted_for_run)
    sft_eval_dataset = (
        strip_to_text_dataset(sft_eval_formatted_for_run)
        if sft_eval_formatted_for_run is not None and len(sft_eval_formatted_for_run) > 0
        else None
    )

    sft_task_loss_datasets: dict[str, Dataset] = {}
    if ENABLE_TASK_LOSS_TENSORBOARD:
        print("Preparation des datasets de loss par tache...", flush=True)
        sft_task_loss_datasets = make_task_loss_datasets(
            sft_train_formatted_for_run,
            split_name="train",
            max_rows=SFT_TASK_LOSS_ROWS,
        )
        if sft_eval_formatted_for_run is not None:
            sft_task_loss_datasets.update(
                make_task_loss_datasets(
                    sft_eval_formatted_for_run,
                    split_name="eval",
                    max_rows=SFT_TASK_LOSS_ROWS,
                )
            )
        print(f"Datasets task loss prets: {list(sft_task_loss_datasets)}", flush=True)
    else:
        print("Task loss TensorBoard desactive: evaluations plus courtes par defaut.", flush=True)

    effective_batch_size = SFT_BATCH_SIZE * SFT_GRAD_ACCUM
    steps_per_epoch = math.ceil(len(sft_train_dataset) / effective_batch_size)
    planned_steps = SFT_MAX_STEPS if SFT_MAX_STEPS is not None else steps_per_epoch * SFT_EPOCHS
    print(f"Exemples SFT train utilises: {len(sft_train_dataset)}", flush=True)
    print(f"Exemples SFT eval utilises : {0 if sft_eval_dataset is None else len(sft_eval_dataset)}", flush=True)
    print(f"Taches train: {task_count_summary(sft_train_formatted_for_run)}", flush=True)
    if sft_eval_formatted_for_run is not None:
        print(f"Taches eval : {task_count_summary(sft_eval_formatted_for_run)}", flush=True)
    print(f"Batch effectif: {effective_batch_size} ({SFT_BATCH_SIZE} x accumulation {SFT_GRAD_ACCUM})", flush=True)
    print(f"Steps/epoch estimes: {steps_per_epoch}", flush=True)
    print(f"Steps planifies: {planned_steps}", flush=True)
    print(f"TensorBoard: {TENSORBOARD_LOG_DIR} sur port {TENSORBOARD_PORT}", flush=True)
    print(f"Task loss detaillee activee: {ENABLE_TASK_LOSS_TENSORBOARD}", flush=True)
    if SFT_MAX_STEPS is not None:
        print("Mode smoke test actif: le modele ne sera pas vraiment fine-tune.", flush=True)

    print("Creation SFTConfig...", flush=True)
    if sft_eval_dataset is None:
        sft_args = make_sft_config(eval_strategy="no", evaluation_strategy="no", do_eval=False)
    else:
        sft_args = make_sft_config()
    print(f"SFTConfig OK en {perf_counter() - t0:.1f}s", flush=True)

    print("Creation data collator...", flush=True)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    print(f"Data collator OK en {perf_counter() - t0:.1f}s", flush=True)

    print("Creation SFTTrainer et tokenisation du dataset...", flush=True)
    sft_trainer = SFTTrainer(
        model=model,
        args=sft_args,
        train_dataset=sft_train_dataset,
        eval_dataset=sft_eval_dataset,
        data_collator=data_collator,
        **trainer_tokenizer_kwarg(SFTTrainer, tokenizer),
    )
    print(f"SFTTrainer OK en {perf_counter() - t0:.1f}s", flush=True)

    print("Masquage des prompts: entrainement uniquement sur les reponses...", flush=True)
    sft_trainer = train_on_responses_only(
        sft_trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    print(f"Masquage OK en {perf_counter() - t0:.1f}s", flush=True)

    if sft_task_loss_datasets:
        sft_trainer.add_callback(TaskLossTensorBoardCallback(TENSORBOARD_LOG_DIR, sft_task_loss_datasets))
        print(f"Loss par tache TensorBoard: {list(sft_task_loss_datasets)}", flush=True)

    sample = sft_trainer.train_dataset[0]
    labels = np.array(sample["labels"])
    trained_tokens = int((labels != -100).sum())
    print(f"Tokens totaux: {len(labels)} | tokens entraines: {trained_tokens}", flush=True)

    if trained_tokens == 0:
        preview = tokenizer.decode(sample["input_ids"][:300])
        raise RuntimeError(
            "Tous les labels sont masques (-100). Le template Qwen3 ou les marqueurs "
            "train_on_responses_only ne correspondent pas. Apercu:\n" + preview
        )
else:
    sft_trainer = None
    print("Creation SFTTrainer ignoree: RUN_SFT = False.")


Creation SFTTrainer ignoree: RUN_SFT = False.


In [9]:
if RUN_SFT and sft_trainer is not None:
    gpu_stats = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu_stats.name}, VRAM totale: {gpu_stats.total_memory / 1e9:.1f} GB")
    print(f"VRAM utilisee avant train: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    sft_trainer_stats = sft_trainer.train()

    print(f"SFT termine en {sft_trainer_stats.metrics.get('train_runtime', 0):.0f}s")
    print(f"Loss finale: {sft_trainer_stats.metrics.get('train_loss', float('nan')):.4f}")
else:
    sft_trainer_stats = None
    print("Entrainement SFT ignore: RUN_SFT = False.")


Entrainement SFT ignore: RUN_SFT = False.


## 7. Test du modele courant

Cette etape est un controle qualitatif minimal sur le modele courant. Elle ne remplace pas une vraie evaluation, mais elle permet de verifier tres vite que le modele charge ou entraine genere encore des reponses coherentes, qu'il respecte le style attendu et qu'il n'a pas completement degrade.

In [10]:

def generate_answer(question: str, max_new_tokens: int = 256) -> str:
    # Petit helper de sanity check pour tester le modele fine-tune sans quitter le notebook.
    FastModel.for_inference(model)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.12,
        no_repeat_ngram_size=5,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    reply = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    # Certaines generations Qwen3 peuvent laisser ce marqueur visible apres decode.
    return reply.replace("<think>\n\n</think>", "").strip()

print(generate_answer("Quels sont les symptomes principaux du diabete de type 2 ?"))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Le diabète de type 2 est une maladie chronique qui affecte l'organisme à cause d'une insuffisance glycémique ou d'un métabolisme inadapté des glucides. Le diabète de Type 2 se caractérise par une augmentation de la production de glucose dans le sang (hyperglycémie) due à une insuffisance fonctionnelle de l'insuline. L’insuline est produite par les glandes sudoraires et régule la sensibilité aux glucoses. La surproduction de glucose dans le plasma sanguin peut être causée par une diminution de la capacité de l'organisme pour absorber cette substance. Une hyperglycémie persistante provoque une dégradation cellulaire progressive, surtout celle des vaisseaux sanguins, ce qui entraîne une altération vasculaire. Cela conduit à une réduction de la circulation sanguine périphérique, donc à une baisse de la disponibilité de l'oxygène dans les tissus. La détérioration de la microcirculation périphérique entraîne une hypoxie des tissus périph


## 8. DPO optionnel

Par defaut `RUN_DPO = False`. Pour lancer seulement le DPO depuis un modele deja fine-tune, mettez `RUN_SFT = False`, `RUN_DPO = True`, puis renseignez `FINETUNED_MODEL_NAME`. Le DPO est plus gourmand en VRAM; sur 8 Go, gardez `DPO_BATCH_SIZE=1`.

La cellule DPO reconditionne les triplets `prompt/chosen/rejected` au format chat, puis lance un entrainement de preference sur le modele courant: soit celui qui vient de passer par le SFT, soit celui charge via `FINETUNED_MODEL_NAME`. Cette etape est optionnelle parce qu'elle ajoute de la complexite experimentale et demande plus de ressources pour un gain qui n'est pas garanti.

In [11]:

def format_dpo_row(example: dict[str, Any]) -> dict[str, str]:
    # Construit un prompt deja template pour que DPO compare seulement les reponses choisie/rejetee.
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": clean_text(example["prompt"])},
    ]
    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    return {
        "prompt": prompt,
        "chosen": direct_answer(example["chosen"]),
        "rejected": direct_answer(example["rejected"]),
    }


if RUN_DPO:
    dpo_raw = load_dataset(DPO_DATASET_ID, split="train")
    require_columns(dpo_raw, {"prompt", "chosen", "rejected"}, DPO_DATASET_ID)

    dpo_dataset = dpo_raw.map(format_dpo_row, remove_columns=dpo_raw.column_names)
    dpo_dataset = dpo_dataset.filter(
        lambda row: bool(row["prompt"].strip()) and bool(row["chosen"].strip()) and bool(row["rejected"].strip())
    )

    print(f"DPO dataset charge: {len(dpo_dataset)} lignes")
    print(dpo_dataset[0])

    FastModel.for_training(model)
    dpo_args = make_dpo_config()

    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=dpo_args,
        train_dataset=dpo_dataset,
        **trainer_tokenizer_kwarg(DPOTrainer, tokenizer),
    )

    dpo_trainer_stats = dpo_trainer.train()
    print(f"DPO termine en {dpo_trainer_stats.metrics.get('train_runtime', 0):.0f}s")
    print(f"Loss finale: {dpo_trainer_stats.metrics.get('train_loss', float('nan')):.4f}")
else:
    print("DPO ignore. Mettre RUN_DPO = True dans la configuration pour l'activer.")


DPO dataset charge: 5000 lignes
{'prompt': '<|im_start|>system\nTu es un assistant medical expert. Reponds de maniere claire, factuelle et structuree. Si la question est en anglais, reponds en anglais.<|im_end|>\n<|im_start|>user\nAnswer the following medical request clearly, factually, and in a structured way. Question: hello dr. i want to ask about i-pill side effects . i had a sex with my boyfriend for 6 times and i took i-pill in 24hours for every time . will it effect me in future. since we r geeting married so for that m not worried but for pregancy m worried.. and how many time we should take i-pill ?<|im_end|>\n<|im_start|>assistant\n', 'chosen': "<think>\n\n</think>\n\nI'm not a doctor, but I can provide you with some general information on the topic. The i-pill, also known as an emergency contraceptive pill (ECP), is intended for occasional use and is not meant to be used as a regular form of birth control. It contains a high dose of levonorgestrel, which can prevent pregnanc

[HAMI-core Msg(4611:124751573333824:multiprocess_memory_limit.c:664)]: Cleanup on exit for PID 4611
[HAMI-core Msg(4611:124751573333824:multiprocess_memory_limit.c:700)]: Exit cleanup complete for PID 4611
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 157
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 17,432,576 of 1,738,007,552 (1.00% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
2,1.792300,24.995502,21.434095,0.671875,3.561408,-489.594940,-400.795776,1.633092,1.656684
4,2.002000,24.841419,21.593164,0.656250,3.248254,-499.327148,-421.438782,1.590313,1.667728
6,1.859000,24.133060,20.976173,0.625000,3.156884,-524.489258,-415.491394,1.634026,1.654172
8,1.558900,25.222073,21.503588,0.671875,3.718482,-521.111633,-408.837524,1.659941,1.708847
10,1.730900,24.792053,21.516096,0.656250,3.275958,-519.972656,-416.506683,1.625485,1.650976
12,1.703900,24.870003,20.683979,0.640625,4.186027,-474.999908,-383.837463,1.584740,1.652750
14,2.201300,26.428082,23.456932,0.687500,2.971151,-524.573608,-434.545746,1.654418,1.711161
16,1.409100,25.249037,21.146776,0.718750,4.102262,-492.908661,-396.719116,1.636922,1.733134
18,2.165300,24.440056,21.484835,0.578125,2.955221,-474.448486,-393.200806,1.583982,1.683872
20,1.391900,25.535442,20.079145,0.734375,5.456295,-552.370605,-416.642212,1.520741,1.455971


DPO termine en 3029s
Loss finale: 1.3634


## 9. Tests finaux

Cette section propose quelques questions de demonstration en francais et en anglais. L'objectif n'est pas de mesurer finement la qualite, mais de verifier la stabilite du modele apres toutes les etapes eventuelles de SFT puis DPO.

In [12]:

for question in [
    "Quels sont les effets secondaires courants des statines ?",
    "What is the recommended first-line treatment for hypertension?",
    "Expliquez la difference entre diabete de type 1 et type 2.",
]:
    print("=" * 80)
    print("Q:", question)
    print("A:", generate_answer(question))


Q: Quels sont les effets secondaires courants des statines ?
A: Les effets secondaires les plus fréquents associés aux statines incluent :

1. **Gout** : Une augmentation du risque de gout (gout de Bouchard) peut survenir chez certains patients.
2. **Dyspepsie** : Cela inclut des troubles digestifs tels que l'irritation gastrique ou le syndrome gastroduodenal.
3. **Cystite** : Augmentation du risque de cystite urinaire.
4. **Mucolaissement** : Des irritations cutanées ou des rougeurs cutanées peuvent survenir.
5. **Toux** : Une toux persistante peut être observée.
6. **Sensibilité à l’acide** : Certains patients ressentent une sensibilité au goût acide.

Il est important de noter que ces effets secondaires sont généralement modérés et qu'ils disparaissent souvent après quelques semaines d'utilisation. Toutefois, ils doivent toujours être surveillés par un médecin.
Q: What is the recommended first-line treatment for hypertension?
A: The American College of Cardiology and the American He

## 10. Sauvegarde et export

La sauvegarde locale exporte les poids LoRA et le tokenizer pour pouvoir recharger rapidement le resultat dans un autre script ou un autre notebook. Le push Hugging Face reste optionnel et est pilote par `PUSH_TO_HUB` afin d'eviter une publication accidentelle pendant les essais.

La cellule suivante montre aussi un exemple d'export GGUF commente. Il sert de pense-bete pour une conversion ulterieure, mais il est volontairement desactive tant que le modele n'a pas ete valide fonctionnellement.

In [13]:

# Sauvegarde locale LoRA.
model.save_pretrained(LOCAL_LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LOCAL_LORA_OUTPUT_DIR)
print(f"LoRA sauvegarde dans {LOCAL_LORA_OUTPUT_DIR}")

if PUSH_TO_HUB:
    model.push_to_hub_merged(
        OUTPUT_HUB_ID,
        tokenizer=tokenizer,
        save_method="merged_16bit",
        token=True,
        private=PRIVATE_HUB_REPO,
    )
    print(f"Modele fusionne pousse: https://huggingface.co/{OUTPUT_HUB_ID}")
else:
    print("Push HF ignore. Mettre PUSH_TO_HUB = True pour exporter le modele fusionne.")


LoRA sauvegarde dans ./qwen3-medical-dpo-lora
Push HF ignore. Mettre PUSH_TO_HUB = True pour exporter le modele fusionne.


In [14]:

# Export GGUF optionnel. A lancer seulement apres validation du modele.
# model.push_to_hub_gguf(
#     f"{OUTPUT_HUB_ID}-GGUF",
#     tokenizer=tokenizer,
#     quantization_method="q4_k_m",
#     token=True,
#     private=PRIVATE_HUB_REPO,
# )
# print(f"GGUF pousse: https://huggingface.co/{OUTPUT_HUB_ID}-GGUF")
